In [ ]:
# TIRO Multi-Agent Contract Redliner

#A 27-agent legal reasoning pipeline that produces redline recommendations
#for commercial contract clauses.

## What This Is

#A single-pass LLM produces one response to one prompt. A TIRO multi-agent
#pipeline decomposes the task into 27 specialized agents across 6 sequential
#rounds, each with a narrow role. The result is a redline that identifies
#more issues, cites more sources, and survives partner-level critique.

## Architecture

#The pipeline follows the TIRO pattern (Trigger, Input, Requirements, Output)
#and the multi-agent legal reasoning architecture described in HAQQ's
#engineering blog:

#- Round 1: Input parsing (4 agents)
#- Round 2: Analysis (8 agents, parallel)
#- Round 3: Playbook application (6 agents, parallel)
#- Round 4: Synthesis (4 agents)
#- Round 5: Verification (2 agents)
#- Round 6: Critique & revision (2 agents)

## Baseline

#The pipeline is compared against a single-pass baseline — the same model,
#the same clause, but one prompt instead of 27.

In [ ]:
# Same stack as Projects 1-3. Groq for the model calls, sentence-transformers
# for embeddings, FAISS for the playbook index. Nothing new here — the
# novelty is in the orchestration, not the infrastructure.
!pip install -q sentence-transformers faiss-cpu groq pandas

import os
import json
import re
import time
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata
from concurrent.futures import ThreadPoolExecutor, as_completed

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

GROQ_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_KEY)

MODEL = "openai/gpt-oss-120b"

print(f"Setup complete. Model: {MODEL}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.2 MB/s eta 0:00:00
Setup complete. Model: openai/gpt-oss-120b


In [ ]:
# This is the core of the multi-agent architecture. Every agent in the
# pipeline uses this same function. The differences between agents are
# entirely in the prompts — the orchestration is uniform.
#
# Design decisions:
#   - Low temperature (0.1) for analytical agents, higher (0.3) for
#     creative agents like the redline drafter.
#   - Max tokens capped per agent. Analytical agents produce short output;
#     synthesis agents produce longer.
#   - Every call is wrapped in retry logic. A 27-agent pipeline will hit
#     rate limits. The retries make it resilient.

def call_agent(system_prompt, user_prompt, agent_name, max_tokens=400, temperature=0.1, max_retries=3):
    """One agent call. Returns the agent's output or an error string."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
                reasoning_effort="low",
            )
            return {
                "agent": agent_name,
                "output": response.choices[0].message.content,
                "success": True,
            }
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(8 * (attempt + 1))
                continue
            return {
                "agent": agent_name,
                "output": f"AGENT FAILED: {str(e)[:100]}",
                "success": False,
            }

# Smoke test. One agent call. If this works, the framework works.
test_result = call_agent(
    system_prompt="You are a legal analyst. Be concise.",
    user_prompt="What is a limitation of liability clause? Answer in one sentence.",
    agent_name="test",
    max_tokens=100,
)
print(f"Test agent: {test_result['success']}")
print(f"Output: {test_result['output'][:150]}")

Test agent: True
Output: A limitation of liability clause is a contract provision that caps or restricts the amount of damages a party can be held responsible for in the event


In [ ]:
# Same two firms as Project 3, same four positions each. Reusing them
# means the multi-agent pipeline can be directly compared against the
# single-pass results from Project 3.
#
# The clause is also the same limitation of liability clause that caused
# Beta's 0% adherence in Project 3. If the multi-agent pipeline fixes
# that, it's a clean demonstration of the architecture's value.

TEST_CLAUSE = {
    "id": "clause_lol",
    "type": "limitation_of_liability",
    "text": "In no event shall either party be liable to the other for any indirect, incidental, special, or consequential damages, whether based on breach of contract, tort, or otherwise. Each party's total aggregate liability under this Agreement shall not exceed the total fees paid by Customer in the twelve (12) months preceding the claim.",
}

PLAYBOOKS = {
    "Alpha LLP": {
        "posture": "Conservative. Risk-averse. Prefers mutual obligations, low caps, broad exit rights. Protecting downside is more important than winning the deal.",
        "positions": {
            "limitation_of_liability": {
                "position": "Mutual cap at 1x fees paid in the prior 12 months. Exclude all consequential, indirect, and special damages.",
                "priority": "High",
                "accept": "2x fees cap with carve-outs only for fraud and wilful misconduct.",
            },
        },
    },
    "Beta Partners": {
        "posture": "Commercial. Aggressive. Prefers one-way protections favoring the client, higher liability caps for counterparty exposure, harder exit provisions. Winning the deal is more important than minimizing downside.",
        "positions": {
            "limitation_of_liability": {
                "position": "Cap at 3x fees paid in the prior 12 months. Carve-outs for IP infringement, data breach, and confidentiality breach — uncapped for those.",
                "priority": "High",
                "accept": "2x cap with IP and data breach carve-outs only.",
            },
        },
    },
}

print(f"Clause loaded: {TEST_CLAUSE['id']} ({TEST_CLAUSE['type']})")
print(f"Firms: {list(PLAYBOOKS.keys())}")
print(f"Both firms have a position on {TEST_CLAUSE['type']}")

Clause loaded: clause_lol (limitation_of_liability)
Firms: ['Alpha LLP', 'Beta Partners']
Both firms have a position on limitation_of_liability


In [ ]:
# Round 1 is sequential in principle but the four agents don't actually
# depend on each other, so I can run them in parallel. Each produces a
# small structured output that the later rounds consume.
#
# Design note: the four agents are narrow. The segmenter doesn't try to
# classify; the classifier doesn't try to extract entities. Narrow roles
# are the whole point of the multi-agent approach — each agent gets a
# prompt tuned to one task, not a generic "analyze this clause" prompt.

def round_1_input_parsing(clause):
    """Break the clause into structured components."""
    print("  Round 1: Input parsing...")

    tasks = [
        {
            "name": "Clause Segmenter",
            "system": "You segment contract clauses into logical units. Return a numbered list of the distinct obligations, conditions, and carve-outs in the clause. Be brief — one line per unit.",
            "user": f"Clause:\n\n{clause['text']}",
            "max_tokens": 250,
        },
        {
            "name": "Entity Extractor",
            "system": "You extract legal entities and defined terms from contract clauses. Return them as a bulleted list. Only include entities and defined terms that appear in the text.",
            "user": f"Clause:\n\n{clause['text']}",
            "max_tokens": 200,
        },
        {
            "name": "Clause Type Classifier",
            "system": "You classify contract clauses by type. Return exactly one of: limitation_of_liability, indemnification, termination, confidentiality, payment, ip_ownership, data_protection, governing_law. Return only the classification, nothing else.",
            "user": f"Clause:\n\n{clause['text']}",
            "max_tokens": 50,
        },
        {
            "name": "Jurisdiction Detector",
            "system": "You identify the governing law of a contract clause when it is evident. If no governing law is specified, return 'unspecified'. Return only the jurisdiction name.",
            "user": f"Clause:\n\n{clause['text']}",
            "max_tokens": 50,
        },
    ]

    results = {}
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {executor.submit(call_agent, t["system"], t["user"], t["name"], t["max_tokens"]): t["name"] for t in tasks}
        for future in as_completed(futures):
            result = future.result()
            results[result["agent"]] = result["output"]
            time.sleep(1)

    return results

r1 = round_1_input_parsing(TEST_CLAUSE)
for name, out in r1.items():
    print(f"    {name}: {out[:100]}...")

  Round 1: Input parsing...
    Clause Type Classifier: limitation_of_liability...
    Entity Extractor: - **party**  
- **Customer**  
- **Agreement**  
- **total fees** (as referenced in the liability li...
    Jurisdiction Detector: unspecified...
    Clause Segmenter: 1. Neither party may be held liable to the other for indirect, incidental, special, or consequential...


In [ ]:
# Round 2 is where the pipeline starts producing value that a single
# prompt can't. Eight specialized agents, each examining one dimension
# of the clause. Run in parallel — they don't depend on each other.
#
# Each agent gets the clause AND the Round 1 output. The context from
# Round 1 makes the analytical agents more precise. In a single-pass
# system, the model has to do all of this in one pass, and it typically
# produces generic observations instead of precise findings.

ANALYSIS_AGENTS = [
    {
        "name": "Liability Cap Analyzer",
        "system": "You analyze the liability cap in a clause. Identify: the cap amount, the basis (fees, contract value, fixed sum), and whether it is mutual or one-way. Be specific.",
    },
    {
        "name": "Carve-out Detector",
        "system": "You identify carve-outs and exclusions to liability caps. Return a bulleted list of every carve-out mentioned, and note which carve-outs are missing that you would expect in a commercial contract.",
    },
    {
        "name": "Consequential Damages Analyzer",
        "system": "You analyze the consequential damages exclusion. Identify which types of damages are excluded (indirect, incidental, special, consequential) and whether the exclusion is mutual.",
    },
    {
        "name": "Mutuality Analyzer",
        "system": "You determine whether each obligation in the clause is mutual (binding both parties) or one-way. Flag any asymmetry and note who it favors.",
    },
    {
        "name": "Scope Limiter Analyzer",
        "system": "You identify any temporal, monetary, or scope-based limitations in the clause (e.g., 'in the 12 months preceding the claim'). List each limiter and what it restricts.",
    },
    {
        "name": "Ambiguity Detector",
        "system": "You identify ambiguous language in the clause — terms that could be interpreted multiple ways, undefined terms, or provisions open to dispute. Be specific about why each is ambiguous.",
    },
    {
        "name": "Missing Provision Detector",
        "system": "You identify provisions that a reasonable counterparty would expect to see in this clause type but that are absent from the text. These are gaps.",
    },
    {
        "name": "Quantitative Exposure Analyzer",
        "system": "You estimate the quantitative exposure created by this clause. If the cap is 'fees paid in 12 months', explain what that means commercially. If carve-outs are uncapped, flag the maximum theoretical exposure.",
    },
]

def round_2_analysis(clause, r1_output):
    print("  Round 2: Analysis (8 agents, parallel)...")
    context = f"Clause:\n{clause['text']}\n\nRound 1 parsing:\n{json.dumps(r1_output, indent=2)}"

    results = {}
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {
            executor.submit(call_agent, a["system"], context, a["name"], 220): a["name"]
            for a in ANALYSIS_AGENTS
        }
        for future in as_completed(futures):
            result = future.result()
            results[result["agent"]] = result["output"]
            time.sleep(1.5)

    return results

r2 = round_2_analysis(TEST_CLAUSE, r1)
for name, out in r2.items():
    print(f"    {name}: {len(out)} chars")

  Round 2: Analysis (8 agents, parallel)...
    Liability Cap Analyzer: 861 chars
    Consequential Damages Analyzer: 789 chars
    Carve-out Detector: 753 chars
    Mutuality Analyzer: 950 chars
    Ambiguity Detector: 1066 chars
    Scope Limiter Analyzer: 920 chars
    Missing Provision Detector: 914 chars
    Quantitative Exposure Analyzer: 735 chars


In [ ]:
# Round 3 is where the twins diverge. Three agents per firm: one matches
# the clause against the firm's primary position, one against the fallback,
# and one scores the priority. Same clause, different playbook.
#
# Six agents, three per firm. Run in parallel. Each agent sees only its
# own firm's playbook — that's what makes the twins distinct.

def round_3_playbook_application(clause, r2_output, firm_name, firm_data):
    print(f"  Round 3: Playbook application for {firm_name}...")
    position = firm_data["positions"].get(clause["type"], {})
    context = f"""Firm: {firm_name}
Posture: {firm_data['posture']}

Playbook position on {clause['type']}:
- Position: {position.get('position', 'not specified')}
- Fallback: {position.get('accept', 'not specified')}
- Priority: {position.get('priority', 'not specified')}

Clause under review:
{clause['text']}

Analysis (from Round 2):
{json.dumps(r2_output, indent=2)}"""

    tasks = [
        {
            "name": f"{firm_name} — Position Matcher",
            "system": "You compare the clause against the firm's primary position. State whether the clause matches, exceeds, or falls short of the position. Be specific about the gap.",
        },
        {
            "name": f"{firm_name} — Fallback Matcher",
            "system": "You compare the clause against the firm's fallback position. State whether the clause would be acceptable as a fallback, and if not, why.",
        },
        {
            "name": f"{firm_name} — Priority Scorer",
            "system": "You score how important this clause is to the firm based on the stated priority. Return: priority (High/Medium/Low), rationale in 2 sentences, and whether the firm should accept, push back, or reject.",
        },
    ]

    results = {}
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {
            executor.submit(call_agent, t["system"], context, t["name"], 250): t["name"]
            for t in tasks
        }
        for future in as_completed(futures):
            result = future.result()
            results[result["agent"]] = result["output"]
            time.sleep(1.5)

    return results

r3_alpha = round_3_playbook_application(TEST_CLAUSE, r2, "Alpha LLP", PLAYBOOKS["Alpha LLP"])
print()
r3_beta = round_3_playbook_application(TEST_CLAUSE, r2, "Beta Partners", PLAYBOOKS["Beta Partners"])

  Round 3: Playbook application for Alpha LLP...

  Round 3: Playbook application for Beta Partners...


In [ ]:
# Round 4 is where the pipeline produces a deliverable. Four agents,
# each producing a different part of the final redline:
#   - Redline Drafter: the actual changes proposed
#   - Negotiation Strategist: how to negotiate them
#   - Risk Scorer: quantifies the exposure if the clause is accepted as-is
#   - Priority Ranker: orders the issues by materiality
#
# These are sequential within a firm because each builds on the prior,
# but Alpha and Beta can run in parallel.

def round_4_synthesis(clause, firm_name, r2_output, r3_output):
    print(f"  Round 4: Synthesis for {firm_name}...")
    context = f"""Firm: {firm_name}

Clause:
{clause['text']}

Analysis:
{json.dumps(r2_output, indent=2)}

Playbook application:
{json.dumps(r3_output, indent=2)}"""

    # Sequential within a firm — the drafter's output feeds the strategist,
    # which feeds the risk scorer, which feeds the ranker.
    draft = call_agent(
        "You draft specific redline changes to a contract clause. For each change, state: what to change, the specific wording you propose, and why. Be precise enough that a lawyer could accept the tracked change.",
        context,
        f"{firm_name} — Redline Drafter",
        max_tokens=600,
        temperature=0.3,
    )
    time.sleep(2)

    strategy = call_agent(
        "You advise on negotiation strategy. Given the clause and the proposed redline, explain how to sequence the negotiation: what to concede first, what to hold firm on, what to trade.",
        f"{context}\n\nProposed redline:\n{draft['output']}",
        f"{firm_name} — Negotiation Strategist",
        max_tokens=400,
        temperature=0.2,
    )
    time.sleep(2)

    risk = call_agent(
        "You quantify commercial risk. Given the clause and analysis, estimate the exposure if the clause is accepted without changes. Give a range and state the assumptions.",
        context,
        f"{firm_name} — Risk Scorer",
        max_tokens=300,
        temperature=0.1,
    )
    time.sleep(2)

    ranking = call_agent(
        "You rank issues by materiality. Given all prior analysis, list the top issues in priority order with a one-line rationale for each ranking.",
        f"{context}\n\nRedline:\n{draft['output']}\n\nStrategy:\n{strategy['output']}\n\nRisk:\n{risk['output']}",
        f"{firm_name} — Priority Ranker",
        max_tokens=300,
        temperature=0.1,
    )

    return {
        "redline": draft["output"],
        "strategy": strategy["output"],
        "risk": risk["output"],
        "ranking": ranking["output"],
    }

r4_alpha = round_4_synthesis(TEST_CLAUSE, "Alpha LLP", r2, r3_alpha)
print()
r4_beta = round_4_synthesis(TEST_CLAUSE, "Beta Partners", r2, r3_beta)

  Round 4: Synthesis for Alpha LLP...

  Round 4: Synthesis for Beta Partners...


In [ ]:
# Rounds 5 and 6 are what separate a good pipeline from a trustworthy
# one. The verification agents check the draft for errors before it
# reaches the user. The critique agents simulate a senior partner
# reviewing the work.
#
# This is the pattern HAQQ's blog describes: verification rounds that
# check every claim and every citation, then a critique round that
# pulls the output apart before it ships.

def extract_json(text):
    if not text:
        return None
    text = re.sub(r'```(?:json)?\s*', '', text).replace('```', '')
    start = text.find('{')
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None


def round_5_verification(clause, firm_name, r4_output):
    print(f"  Round 5: Verification for {firm_name}...")
    context = f"""Clause:
{clause['text']}

Draft redline:
{r4_output['redline']}

Strategy:
{r4_output['strategy']}"""

    verifier = call_agent(
        "You verify a legal draft. Check every factual claim, every cited provision, and every asserted rule. Return a json object with: verified (bool), unverified_claims (list), unverified_citations (list), issues (list).",
        context,
        f"{firm_name} — Citation Verifier",
        max_tokens=350,
        temperature=0.0,
    )
    time.sleep(2)

    consistency = call_agent(
        "You check internal consistency of a legal draft. Flag any contradictions between the redline, the strategy, and the analysis. Return a json object with: consistent (bool), contradictions (list), ambiguities (list).",
        context,
        f"{firm_name} — Consistency Checker",
        max_tokens=300,
        temperature=0.0,
    )

    return {
        "citation_verification": verifier["output"],
        "consistency_check": consistency["output"],
    }


def round_6_critique_and_revision(clause, firm_name, r4_output, r5_output):
    print(f"  Round 6: Critique and revision for {firm_name}...")
    context = f"""Clause:
{clause['text']}

Draft:
{r4_output['redline']}

Verification:
{json.dumps(r5_output, indent=2)}"""

    critique = call_agent(
        "You are a senior partner reviewing an associate's redline. Be direct. Identify weaknesses: what did they miss? What is too aggressive? What is too soft? What would you change? Return 3-5 specific critiques.",
        context,
        f"{firm_name} — Partner Simulator",
        max_tokens=400,
        temperature=0.3,
    )
    time.sleep(2)

    revision = call_agent(
        "You revise a legal draft based on a senior partner's critique. Produce the final version. Address each critique explicitly and note what you changed and why.",
        f"{context}\n\nPartner critique:\n{critique['output']}",
        f"{firm_name} — Revision Agent",
        max_tokens=600,
        temperature=0.2,
    )

    return {
        "critique": critique["output"],
        "revised": revision["output"],
    }

r5_alpha = round_5_verification(TEST_CLAUSE, "Alpha LLP", r4_alpha)
r6_alpha = round_6_critique_and_revision(TEST_CLAUSE, "Alpha LLP", r4_alpha, r5_alpha)
print()
r5_beta = round_5_verification(TEST_CLAUSE, "Beta Partners", r4_beta)
r6_beta = round_6_critique_and_revision(TEST_CLAUSE, "Beta Partners", r4_beta, r5_beta)

  Round 5: Verification for Alpha LLP...
  Round 6: Critique and revision for Alpha LLP...

  Round 5: Verification for Beta Partners...
  Round 6: Critique and revision for Beta Partners...


In [ ]:
# The baseline. One prompt, one agent, one response. This is what the
# pipeline is being compared against.
#
# Same model, same clause, same playbook. The only difference is the
# architecture. If the pipeline outperforms this, it is because of the
# decomposition, not because of a better model.

def single_pass_baseline(clause, firm_name, firm_data):
    """Single-pass generation. One agent, one prompt, one response."""
    position = firm_data["positions"].get(clause["type"], {})
    prompt = f"""You are the legal AI assistant for {firm_name}.

Firm posture: {firm_data['posture']}

Playbook position on {clause['type']}:
{position.get('position', 'not specified')}

Fallback: {position.get('accept', 'not specified')}

Clause to review:
{clause['text']}

Produce a redline recommendation with:
1. Specific changes to propose
2. Reasoning grounded in the playbook
3. Negotiation strategy"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=600,
        reasoning_effort="low",
    )
    return response.choices[0].message.content

print("Running single-pass baseline...")
baseline_alpha = single_pass_baseline(TEST_CLAUSE, "Alpha LLP", PLAYBOOKS["Alpha LLP"])
time.sleep(3)
baseline_beta = single_pass_baseline(TEST_CLAUSE, "Beta Partners", PLAYBOOKS["Beta Partners"])

print(f"\nBaseline Alpha output length: {len(baseline_alpha)} chars")
print(f"Baseline Beta output length: {len(baseline_beta)} chars")

Running single-pass baseline...

Baseline Alpha output length: 2432 chars
Baseline Beta output length: 2599 chars


In [ ]:
# Forgot this in the setup. Same embedder as Projects 1-3. Only needed
# for the divergence comparison at the end, which is why it wasn't
# loaded earlier — the pipeline itself doesn't use embeddings.
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("Embedder loaded.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded.


In [ ]:
# The comparison. Count citations, count proposed changes, measure
# output length, and compute divergence. HAQQ's blog claims multi-agent
# produces 3.9x more track changes and 18 citations vs zero. I want to
# see if my pipeline shows a similar pattern.

def count_citations(text):
    """Rough heuristic: count references to articles, sections, provisions."""
    if not text:
        return 0
    patterns = [
        r'\bArticle\s+\d+',
        r'\bSection\s+\d+',
        r'\bClause\s+\d+',
        r'\bDecree[- ]Law\b',
        r'\bLaw\s+No\.',
        r'\bRegulations?\s+\d{4}',
    ]
    return sum(len(re.findall(p, text, re.IGNORECASE)) for p in patterns)


def count_track_changes(text):
    """Count proposed changes — sentences with imperative language."""
    if not text:
        return 0
    indicators = [
        r'\bshall\s+be\s+(amended|revised|changed|modified|deleted|inserted)',
        r'\breplace\s',
        r'\bdelete\s',
        r'\binsert\s',
        r'\bchange\s',
        r'\bpropose\s',
        r'\bamend\s',
        r'\badd\s',
    ]
    return sum(len(re.findall(p, text, re.IGNORECASE)) for p in indicators)


# Aggregate outputs for comparison
def aggregate_pipeline_output(r4, r6):
    return "\n\n".join([r4["redline"], r4["strategy"], r6["revised"]])

pipeline_alpha = aggregate_pipeline_output(r4_alpha, r6_alpha)
pipeline_beta = aggregate_pipeline_output(r4_beta, r6_beta)

comparison = pd.DataFrame([
    {
        "condition": "Baseline (Alpha)",
        "chars": len(baseline_alpha),
        "citations": count_citations(baseline_alpha),
        "track_changes": count_track_changes(baseline_alpha),
    },
    {
        "condition": "Pipeline (Alpha)",
        "chars": len(pipeline_alpha),
        "citations": count_citations(pipeline_alpha),
        "track_changes": count_track_changes(pipeline_alpha),
    },
    {
        "condition": "Baseline (Beta)",
        "chars": len(baseline_beta),
        "citations": count_citations(baseline_beta),
        "track_changes": count_track_changes(baseline_beta),
    },
    {
        "condition": "Pipeline (Beta)",
        "chars": len(pipeline_beta),
        "citations": count_citations(pipeline_beta),
        "track_changes": count_track_changes(pipeline_beta),
    },
])

print("=" * 90)
print("SINGLE-PASS vs MULTI-AGENT PIPELINE")
print("=" * 90)
print(comparison.to_string(index=False))
print("=" * 90)

# Divergence between Alpha and Beta under each condition
a_base_emb = model.encode([baseline_alpha], normalize_embeddings=True)
b_base_emb = model.encode([baseline_beta], normalize_embeddings=True)
base_sim = float(np.dot(a_base_emb[0], b_base_emb[0]))

a_pipe_emb = model.encode([pipeline_alpha], normalize_embeddings=True)
b_pipe_emb = model.encode([pipeline_beta], normalize_embeddings=True)
pipe_sim = float(np.dot(a_pipe_emb[0], b_pipe_emb[0]))

print(f"\nAlpha vs Beta similarity — Baseline: {base_sim:.3f}")
print(f"Alpha vs Beta similarity — Pipeline: {pipe_sim:.3f}")
print(f"(Lower similarity = more divergent output between firms)")

SINGLE-PASS vs MULTI-AGENT PIPELINE
       condition  chars  citations  track_changes
Baseline (Alpha)   2432          0              3
Pipeline (Alpha)   6302          1              5
 Baseline (Beta)   2599          0              3
 Pipeline (Beta)   6628          0             10

Alpha vs Beta similarity — Baseline: 0.709
Alpha vs Beta similarity — Pipeline: 0.746
(Lower similarity = more divergent output between firms)


In [ ]:
comparison.to_csv("tiro_comparison.csv", index=False)

summary = {
    "baseline_alpha_chars": len(baseline_alpha),
    "pipeline_alpha_chars": len(pipeline_alpha),
    "baseline_alpha_citations": count_citations(baseline_alpha),
    "pipeline_alpha_citations": count_citations(pipeline_alpha),
    "baseline_alpha_track_changes": count_track_changes(baseline_alpha),
    "pipeline_alpha_track_changes": count_track_changes(pipeline_alpha),
    "baseline_beta_chars": len(baseline_beta),
    "pipeline_beta_chars": len(pipeline_beta),
    "baseline_beta_citations": count_citations(baseline_beta),
    "pipeline_beta_citations": count_citations(pipeline_beta),
    "baseline_beta_track_changes": count_track_changes(baseline_beta),
    "pipeline_beta_track_changes": count_track_changes(pipeline_beta),
    "baseline_similarity": round(base_sim, 3),
    "pipeline_similarity": round(pipe_sim, 3),
    "n_agents": 27,
    "n_rounds": 6,
}
with open("tiro_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved tiro_comparison.csv and tiro_summary.json")
print(json.dumps(summary, indent=2))

Saved tiro_comparison.csv and tiro_summary.json
{
  "baseline_alpha_chars": 2432,
  "pipeline_alpha_chars": 6302,
  "baseline_alpha_citations": 0,
  "pipeline_alpha_citations": 1,
  "baseline_alpha_track_changes": 3,
  "pipeline_alpha_track_changes": 5,
  "baseline_beta_chars": 2599,
  "pipeline_beta_chars": 6628,
  "baseline_beta_citations": 0,
  "pipeline_beta_citations": 0,
  "baseline_beta_track_changes": 3,
  "pipeline_beta_track_changes": 10,
  "baseline_similarity": 0.709,
  "pipeline_similarity": 0.746,
  "n_agents": 27,
  "n_rounds": 6
}


In [ ]:
# The citation counter looks for specific patterns. The pipeline might
# be citing provisions in a different format, or not citing at all.
# I need to see what it actually produced before writing the README.

print("PIPELINE ALPHA — first 2000 chars")
print("=" * 80)
print(pipeline_alpha[:2000])
print("\n\nPIPELINE BETA — first 2000 chars")
print("=" * 80)
print(pipeline_beta[:2000])

PIPELINE ALPHA — first 2000 chars
**Red‑line Recommendations for the Limitation‑of‑Liability Clause**  
*(Prepared for Alpha LLP – “high‑priority” play‑book)*  

| # | What to change | Proposed wording (track‑change style) | Why – legal/risk rationale |
|---|----------------|----------------------------------------|----------------------------|
| 1 | **Increase the liability‑cap multiplier to the firm’s fallback of 2 × fees paid** | “…shall not exceed **twice** the total fees paid by Customer in the twelve (12) months preceding the claim.” | Alpha LLP’s fallback position requires a higher ceiling (2 ×) to protect against under‑insurance. A 2 × multiplier is common in SaaS and professional‑services agreements and is more likely to survive a “unconscionability” challenge. |
| 2 | **Make the cap symmetric (each party capped by the fees it has paid)** | “Each party’s total aggregate liability under this Agreement shall not exceed **the greater of** (i) the total fees paid by that party in 

In [ ]:
with open("clause.json", "w", encoding="utf-8") as f:
    json.dump(TEST_CLAUSE, f, ensure_ascii=False, indent=2)

with open("playbooks.json", "w", encoding="utf-8") as f:
    json.dump(PLAYBOOKS, f, ensure_ascii=False, indent=2)

print("Wrote clause.json and playbooks.json")

Wrote clause.json and playbooks.json
